## SRP466358

**paper:** [PMID: 38988981](https://pmc.ncbi.nlm.nih.gov/articles/PMC11233799/) - Lipidomic and transcriptomic profiles provide new insights into the triacylglycerol and glucose handling capacities of the Arctic fox, 2024

**date, curator:** 2026-01-28, Sara Carsanaro

**notes**
* low fat vs high fat diet
    * LFA = 15% crude fat diet
    * HFA = 40% crude fat diet
* SRA says animals were 8 months old, however
    * per methods: Each animal was individually housed, and following a one-week adaptation period, then they were fed each diet for 11 weeks -> 3 months
    * not clear if this started at 5 months and they ended at 8 months, or if they started at 8 months and ended at 11 months 
    * of course, fox sexual maturity is around 10 months
    * to be safe, will annotate age with fully formed since it isn't clear if they are sexually mature or not (although if i had to guess, i would guess immature)
* for red fox (tax id 9627), strain = Silver (strain is in uniprot)

### annotation summary
run this after annotation is complete

In [24]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,liver,UBERON:0002107,liver,perfect match


In [25]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,8 months,UBERON:0000066,fully formed stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP466358"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 13/13 [00:12<00:00,  1.06it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes
Fetching RunIds and ReadHashes for each library...can take several minutes


### library annnotations

In [7]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX22092763,SRP466358,Illumina NovaSeq 6000,SRS19158724,,,,,,liver,8 months,,,,M,,,494514,,,,,,RNA-seq of liver3 of HFB,SAMN37813341,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver3 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386635,FC31DB00ACDB44F549A8CB8243B7CDED
1,SRX22092762,SRP466358,Illumina NovaSeq 6000,SRS19158723,,,,,,liver,8 months,,,,M,,,494514,,,,,,RNA-seq of liver2 of HFB,SAMN37813340,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386636,8B01095C962F67982850E79374CD7C2A
2,SRX22092761,SRP466358,Illumina NovaSeq 6000,SRS19158722,,,,,,liver,8 months,,,,M,Silver,,9627,,,,,,RNA-seq of liver2 of LFS,SAMN37813331,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386637,F3F519AC02C16F36D7A50BCBD1D9A1E6
3,SRX22092760,SRP466358,Illumina NovaSeq 6000,SRS19158721,,,,,,liver,8 months,,,,M,Silver,,9627,,,,,,RNA-seq of liver1 of LFS,SAMN37813330,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386638,CFE34094EA96091692B24AC4F8F4B09F
4,SRX22092759,SRP466358,Illumina NovaSeq 6000,SRS19158720,,,,,,liver,8 months,,,,M,,,494514,,,,,,RNA-seq of liver1 of HFB,SAMN37813339,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386639,84D7A7F64F263CACC3A2EA0BD10E48BE
5,SRX22092758,SRP466358,Illumina NovaSeq 6000,SRS19158719,,,,,,liver,8 months,,,,M,,,494514,,,,,,RNA-seq of liver5 of LFB,SAMN37813338,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver5 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386640,602B66FDC279F10D15D3D5A00916E884
6,SRX22092757,SRP466358,Illumina NovaSeq 6000,SRS19158718,,,,,,liver,8 months,,,,M,,,494514,,,,,,RNA-seq of liver4 of LFB,SAMN37813337,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver4 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386641,7CF209BDA4D95A3C583C9E83ADC32D16
7,SRX22092756,SRP466358,Illumina NovaSeq 6000,SRS19158717,,,,,,liver,8 months,,,,M,,,494514,,,,,,RNA-seq of liver3 of LFB,SAMN37813336,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver3 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386642,607D303248E899063D2257AF5D14315D
8,SRX22092755,SRP466358,Illumina NovaSeq 6000,SRS19158716,,,,,,liver,8 months,,,,M,,,494514,,,,,,RNA-seq of liver2 of LFB,SAMN37813335,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386643,9C3300E6B075F333996508F4F03616B1
9,SRX22092754,SRP466358,Illumina NovaSeq 6000,SRS19158715,,,,,,liver,8 months,,,,M,,,494514,,,,,,RNA-seq of liver1 of LFB,SAMN37813334,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of LFB,,,,

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [8]:
unique_sorted(library, "infoOrgan")

['liver']


In [9]:

# all
library.loc[:,'anatId'] = 'UBERON:0002107'
library.loc[:,'anatName'] = 'liver'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX22092763,SRP466358,Illumina NovaSeq 6000,SRS19158724,UBERON:0002107,liver,,,,liver,8 months,perfect match,not documented,,M,,,494514,,,,,,RNA-seq of liver3 of HFB,SAMN37813341,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver3 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386635,FC31DB00ACDB44F549A8CB8243B7CDED
1,SRX22092762,SRP466358,Illumina NovaSeq 6000,SRS19158723,UBERON:0002107,liver,,,,liver,8 months,perfect match,not documented,,M,,,494514,,,,,,RNA-seq of liver2 of HFB,SAMN37813340,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386636,8B01095C962F67982850E79374CD7C2A
2,SRX22092761,SRP466358,Illumina NovaSeq 6000,SRS19158722,UBERON:0002107,liver,,,,liver,8 months,perfect match,not documented,,M,Silver,,9627,,,,,,RNA-seq of liver2 of LFS,SAMN37813331,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386637,F3F519AC02C16F36D7A50BCBD1D9A1E6
3,SRX22092760,SRP466358,Illumina NovaSeq 6000,SRS19158721,UBERON:0002107,liver,,,,liver,8 months,perfect match,not documented,,M,Silver,,9627,,,,,,RNA-seq of liver1 of LFS,SAMN37813330,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386638,CFE34094EA96091692B24AC4F8F4B09F
4,SRX22092759,SRP466358,Illumina NovaSeq 6000,SRS19158720,UBERON:0002107,liver,,,,liver,8 months,perfect match,not documented,,M,,,494514,,,,,,RNA-seq of liver1 of HFB,SAMN37813339,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386639,84D7A7F64F263CACC3A2EA0BD10E48BE
5,SRX22092758,SRP466358,Illumina NovaSeq 6000,SRS19158719,UBERON:0002107,liver,,,,liver,8 months,perfect match,not documented,,M,,,494514,,,,,,RNA-seq of liver5 of LFB,SAMN37813338,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver5 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386640,602B66FDC279F10D15D3D5A00916E884
6,SRX22092757,SRP466358,Illumina NovaSeq 6000,SRS19158718,UBERON:0002107,liver,,,,liver,8 months,perfect match,not documented,,M,,,494514,,,,,,RNA-seq of liver4 of LFB,SAMN37813337,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver4 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386641,7CF209BDA4D95A3C583C9E83ADC32D16
7,SRX22092756,SRP466358,Illumina NovaSeq 6000,SRS19158717,UBERON:0002107,liver,,,,liver,8 months,perfect match,not documented,,M,,,494514,,,,,,RNA-seq of liver3 of LFB,SAMN37813336,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver3 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386642,607D303248E899063D2257AF5D14315D
8,SRX22092755,SRP466358,Illumina NovaSeq 6000,SRS19158716,UBERON:0002107,liver,,,,liver,8 months,perfect match,not documented,,M,,,494514,,,,,,RNA-seq of liver2 of LFB,SAMN37813335,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

using fully formed stage here because not clear if they are sexually mature or immature 

In [10]:
unique_sorted(library, "infoStage")

['8 months']


In [11]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'other'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX22092763,SRP466358,Illumina NovaSeq 6000,SRS19158724,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,,,,,,RNA-seq of liver3 of HFB,SAMN37813341,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver3 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386635,FC31DB00ACDB44F549A8CB8243B7CDED
1,SRX22092762,SRP466358,Illumina NovaSeq 6000,SRS19158723,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,,,,,,RNA-seq of liver2 of HFB,SAMN37813340,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386636,8B01095C962F67982850E79374CD7C2A
2,SRX22092761,SRP466358,Illumina NovaSeq 6000,SRS19158722,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,,,,,,RNA-seq of liver2 of LFS,SAMN37813331,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386637,F3F519AC02C16F36D7A50BCBD1D9A1E6
3,SRX22092760,SRP466358,Illumina NovaSeq 6000,SRS19158721,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,,,,,,RNA-seq of liver1 of LFS,SAMN37813330,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386638,CFE34094EA96091692B24AC4F8F4B09F
4,SRX22092759,SRP466358,Illumina NovaSeq 6000,SRS19158720,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,,,,,,RNA-seq of liver1 of HFB,SAMN37813339,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386639,84D7A7F64F263CACC3A2EA0BD10E48BE
5,SRX22092758,SRP466358,Illumina NovaSeq 6000,SRS19158719,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,,,,,,RNA-seq of liver5 of LFB,SAMN37813338,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver5 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386640,602B66FDC279F10D15D3D5A00916E884
6,SRX22092757,SRP466358,Illumina NovaSeq 6000,SRS19158718,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,,,,,,RNA-seq of liver4 of LFB,SAMN37813337,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver4 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386641,7CF209BDA4D95A3C583C9E83ADC32D16
7,SRX22092756,SRP466358,Illumina NovaSeq 6000,SRS19158717,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,,,,,,RNA-seq of liver3 of LFB,SAMN37813336,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver3 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386642,

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = 'Silver' -> did manually in the end

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [12]:
# making these variables because we use them again in the experiment file
my_protocol = 'VAHTS Universal V6 RNA-seq Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX22092763,SRP466358,Illumina NovaSeq 6000,SRS19158724,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver3 of HFB,SAMN37813341,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver3 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386635,FC31DB00ACDB44F549A8CB8243B7CDED
1,SRX22092762,SRP466358,Illumina NovaSeq 6000,SRS19158723,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of HFB,SAMN37813340,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386636,8B01095C962F67982850E79374CD7C2A
2,SRX22092761,SRP466358,Illumina NovaSeq 6000,SRS19158722,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of LFS,SAMN37813331,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386637,F3F519AC02C16F36D7A50BCBD1D9A1E6
3,SRX22092760,SRP466358,Illumina NovaSeq 6000,SRS19158721,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of LFS,SAMN37813330,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386638,CFE34094EA96091692B24AC4F8F4B09F
4,SRX22092759,SRP466358,Illumina NovaSeq 6000,SRS19158720,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of HFB,SAMN37813339,,,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386639,84D7A7F64F263CACC3A2EA0BD10E48BE
5,SRX22092758,SRP466358,Illumina NovaSeq 6000,SRS19158719,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver5 of LFB,SAMN37813338,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver5 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386640,602B66FDC279F10D15D3D5A00916E884
6,SRX22092757,SRP466358,Illumina NovaSeq 6000,SRS19158718,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver4 of LFB,SAMN37813337,,,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver4 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386641,7CF209BDA4D95A3C583

#### globin, replicates

In [13]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [14]:
library.loc[:,'sampleAge_value'] = '8'
library.loc[:,'sampleAge_unit'] = 'month'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX22092763,SRP466358,Illumina NovaSeq 6000,SRS19158724,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver3 of HFB,SAMN37813341,8,month,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver3 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386635,FC31DB00ACDB44F549A8CB8243B7CDED
1,SRX22092762,SRP466358,Illumina NovaSeq 6000,SRS19158723,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of HFB,SAMN37813340,8,month,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386636,8B01095C962F67982850E79374CD7C2A
2,SRX22092761,SRP466358,Illumina NovaSeq 6000,SRS19158722,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of LFS,SAMN37813331,8,month,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386637,F3F519AC02C16F36D7A50BCBD1D9A1E6
3,SRX22092760,SRP466358,Illumina NovaSeq 6000,SRS19158721,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of LFS,SAMN37813330,8,month,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386638,CFE34094EA96091692B24AC4F8F4B09F
4,SRX22092759,SRP466358,Illumina NovaSeq 6000,SRS19158720,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of HFB,SAMN37813339,8,month,,,,,,HF: 40% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386639,84D7A7F64F263CACC3A2EA0BD10E48BE
5,SRX22092758,SRP466358,Illumina NovaSeq 6000,SRS19158719,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver5 of LFB,SAMN37813338,8,month,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver5 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386640,602B66FDC279F10D15D3D5A00916E884
6,SRX22092757,SRP466358,Illumina NovaSeq 6000,SRS19158718,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver4 of LFB,SAMN37813337,8,month,,,,,,LF: 15% crude fat diet,,,28/01/2026,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver4 of LFB,,,,,,TRANSCRIP

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [15]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-01-28'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX22092763,SRP466358,Illumina NovaSeq 6000,SRS19158724,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver3 of HFB,SAMN37813341,8,month,,,,,,HF: 40% crude fat diet,,SAC,2026-01-28,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver3 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386635,FC31DB00ACDB44F549A8CB8243B7CDED
1,SRX22092762,SRP466358,Illumina NovaSeq 6000,SRS19158723,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of HFB,SAMN37813340,8,month,,,,,,HF: 40% crude fat diet,,SAC,2026-01-28,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386636,8B01095C962F67982850E79374CD7C2A
2,SRX22092761,SRP466358,Illumina NovaSeq 6000,SRS19158722,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of LFS,SAMN37813331,8,month,,,,,,LF: 15% crude fat diet,,SAC,2026-01-28,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver2 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386637,F3F519AC02C16F36D7A50BCBD1D9A1E6
3,SRX22092760,SRP466358,Illumina NovaSeq 6000,SRS19158721,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of LFS,SAMN37813330,8,month,,,,,,LF: 15% crude fat diet,,SAC,2026-01-28,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of LFS,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386638,CFE34094EA96091692B24AC4F8F4B09F
4,SRX22092759,SRP466358,Illumina NovaSeq 6000,SRS19158720,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of HFB,SAMN37813339,8,month,,,,,,HF: 40% crude fat diet,,SAC,2026-01-28,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver1 of HFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386639,84D7A7F64F263CACC3A2EA0BD10E48BE
5,SRX22092758,SRP466358,Illumina NovaSeq 6000,SRS19158719,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver5 of LFB,SAMN37813338,8,month,,,,,,LF: 15% crude fat diet,,SAC,2026-01-28,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver5 of LFB,,,,,,TRANSCRIPTOMIC,cDNA,SRR26386640,602B66FDC279F10D15D3D5A00916E884
6,SRX22092757,SRP466358,Illumina NovaSeq 6000,SRS19158718,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver4 of LFB,SAMN37813337,8,month,,,,,,LF: 15% crude fat diet,,SAC,2026-01-28,"Total RNA of liver was extracted using RNeasy Mini Kit (QIAGEN, CA, USA)",,RNA-seq of liver4 

#### comments

In [16]:
library.loc[:,'comment'] = 'PMID: 38988981'

#### save complete file with correct columns

In [17]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX22092763,SRP466358,Illumina NovaSeq 6000,SRS19158724,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver3 of HFB,SAMN37813341,8,month,,,,,PMID: 38988981,HF: 40% crude fat diet,,SAC,2026-01-28
1,SRX22092762,SRP466358,Illumina NovaSeq 6000,SRS19158723,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of HFB,SAMN37813340,8,month,,,,,PMID: 38988981,HF: 40% crude fat diet,,SAC,2026-01-28
2,SRX22092761,SRP466358,Illumina NovaSeq 6000,SRS19158722,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of LFS,SAMN37813331,8,month,,,,,PMID: 38988981,LF: 15% crude fat diet,,SAC,2026-01-28
3,SRX22092760,SRP466358,Illumina NovaSeq 6000,SRS19158721,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of LFS,SAMN37813330,8,month,,,,,PMID: 38988981,LF: 15% crude fat diet,,SAC,2026-01-28
4,SRX22092759,SRP466358,Illumina NovaSeq 6000,SRS19158720,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of HFB,SAMN37813339,8,month,,,,,PMID: 38988981,HF: 40% crude fat diet,,SAC,2026-01-28
5,SRX22092758,SRP466358,Illumina NovaSeq 6000,SRS19158719,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver5 of LFB,SAMN37813338,8,month,,,,,PMID: 38988981,LF: 15% crude fat diet,,SAC,2026-01-28
6,SRX22092757,SRP466358,Illumina NovaSeq 6000,SRS19158718,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver4 of LFB,SAMN37813337,8,month,,,,,PMID: 38988981,LF: 15% crude fat diet,,SAC,2026-01-28
7,SRX22092756,SRP466358,Illumina NovaSeq 6000,SRS19158717,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver3 of LFB,SAMN37813336,8,month,,,,,PMID: 38988981,LF: 15% crude fat diet,,SAC,2026-01-28
8,SRX22092755,SRP466358,Illumina NovaSeq 6000,SRS19158716,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of LFB,SAMN37813335,8,month,,,,,PMID: 38988981,LF: 15% crude fat diet,,SAC,2026-01-28
9,SRX22092754,SRP466358,Illumina NovaSeq 6000,SRS19158715,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of LFB,SAMN37813334,8,month,,,,,PMID: 38988981,LF: 15% crude fat diet,,SAC,2026-01-28


### experiment annotations

In [18]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP466358,Arctic fox and silver fox liver transcriptome,The transcriptome of livers of arctic fox and silver fox fed with 15% or 40% fat content diet.,SRA,,,,,,,PRJNA1028137,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [19]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

13

In [20]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP466358,Arctic fox and silver fox liver transcriptome,The transcriptome of livers of arctic fox and silver fox fed with 15% or 40% fat content diet.,SRA,total,Bgee 1K,13,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,,PRJNA1028137,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [21]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '38988981'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11233799/'
experiment.loc[:,'DOI'] = '10.3389/fvets.2024.1388532'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP466358,Arctic fox and silver fox liver transcriptome,The transcriptome of livers of arctic fox and silver fox fed with 15% or 40% fat content diet.,SRA,total,Bgee 1K,13,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,,PRJNA1028137,38988981,https://pmc.ncbi.nlm.nih.gov/articles/PMC11233799/,10.3389/fvets.2024.1388532,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [22]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [23]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [26]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [27]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
53326,SRX11148746,SRP324111,Illumina HiSeq 3000,SRS9210403,CL:0000784,plasmacytoid dendritic cell,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,PBMC - pDC,4 years,perfect match,not documented,missing child term,M,,,9531,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,19-5FKb1-pDC-SM_S112,"SAMN19712285,GSM5384731",4,year,,,,,SIV neg,,,SAC,2026-01-28
53327,SRX11148745,SRP324111,Illumina HiSeq 3000,SRS9210402,CL:0000784,plasmacytoid dendritic cell,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,PBMC - pDC,4 years,perfect match,not documented,missing child term,M,,,9531,SMART-Seq v4 Ultra Low Input,full_length,polyA,,,17-4FJb1-pDC-SM_S110,"SAMN19712286,GSM5384730",4,year,,,,,SIV neg,,,SAC,2026-01-28
53328,SRX22092763,SRP466358,Illumina NovaSeq 6000,SRS19158724,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver3 of HFB,SAMN37813341,8,month,,,,,PMID: 38988981,HF: 40% crude fat diet,,SAC,2026-01-28
53329,SRX22092762,SRP466358,Illumina NovaSeq 6000,SRS19158723,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of HFB,SAMN37813340,8,month,,,,,PMID: 38988981,HF: 40% crude fat diet,,SAC,2026-01-28
53330,SRX22092761,SRP466358,Illumina NovaSeq 6000,SRS19158722,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver2 of LFS,SAMN37813331,8,month,,,,,PMID: 38988981,LF: 15% crude fat diet,,SAC,2026-01-28
53331,SRX22092760,SRP466358,Illumina NovaSeq 6000,SRS19158721,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,Silver,,9627,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of LFS,SAMN37813330,8,month,,,,,PMID: 38988981,LF: 15% crude fat diet,,SAC,2026-01-28
53332,SRX22092759,SRP466358,Illumina NovaSeq 6000,SRS19158720,UBERON:0002107,liver,UBERON:0000066,fully formed stage,,liver,8 months,perfect match,not documented,other,M,,,494514,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,polyA,,,RNA-seq of liver1 of HFB,SAMN37813339,8,month,,,,,PMID: 38988981,HF: 40% crude fat diet,,SAC,2026-01-28


In [28]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1009,SRP313647,Transcriptomic signatures of temperature adapt...,"In this study, we used comparative transcripto...",SRA,total,Bgee 1K,24,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA719567,33837581,https://academic.oup.com/jeb/article/34/8/1212...,10.1111/jeb.13789,,"oyster, adult stage mapped to UBERON:0000113 ..."
1010,SRP324111,Tissue-specific transcriptional profiling of p...,HIV associated immune activation (IA) is assoc...,SRA,partial,Bgee 1K,16,SMART-Seq v4 Ultra Low Input,full_length,GSE178225,PRJNA737767,34181694,https://pmc.ncbi.nlm.nih.gov/articles/PMC8270445/,10.1371/journal.ppat.1009674,,partical b/c removed infected samples - SIV an...
1011,SRP466358,Arctic fox and silver fox liver transcriptome,The transcriptome of livers of arctic fox and ...,SRA,total,Bgee 1K,13,VAHTS Universal V6 RNA-seq Library Prep Kit,full_length,,PRJNA1028137,38988981,https://pmc.ncbi.nlm.nih.gov/articles/PMC11233...,10.3389/fvets.2024.1388532,,


### add annotations to git

In [29]:
! git pull

Already up to date.


In [30]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [31]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../1_template/1_complete_bulk_template.ipynb
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [32]:
! git add $git_experiment_path $git_library_path

In [33]:
! git commit -m $commit_message_exp

[develop 2a2cef7] adding annotated bulk experiment SRP466358
 2 files changed, 14 insertions(+)


In [34]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.98 KiB | 1.98 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   ef78ab0..2a2cef7  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push